# Integers Window (sliding window aggregates)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Arrays, Sliding Window · **Difficulty/Frequency:** Rare (2/10)

## Concepts

**What this problem is really testing:**
- The **sliding window** idea: update an aggregate incrementally instead of recomputing it
- Recognising which aggregates support that, and which **do not**
- Deciding what happens at the boundary, rather than letting the code decide by accident

**First-principles primer — what is each piece?**

- **The sliding window.** Consecutive windows overlap almost completely. Moving from `arr[0:3]` to `arr[1:4]`, only **two** elements change: one leaves the left, one enters the right. Everything in between is untouched — so re-adding it is wasted work.
- **The incremental update.** Keep the sum as state and repair it:

  ```
  sum -= arr[start]        # the element leaving
  sum += arr[start + n]    # the element entering
  ```

  O(n) once in the constructor, then **O(1) forever**.

- **The invariant.** *Before every call, `current_sum` equals the sum of `arr[start : start + n]`.* Say this before writing code and the correctness argument is finished: it holds initially by construction, and each slide subtracts exactly what left and adds exactly what arrived.

**Which aggregates can slide, and which cannot** — this is the part worth understanding rather than memorising:

| Aggregate | Slides? | Why |
|---|---|---|
| **Sum** | ✅ | subtraction undoes addition, exactly |
| **Average** | ✅ | it is just the sum, divided at the end |
| **Product** | ⚠️ | division undoes multiplication — **unless the leaving element is 0** |
| **Min / max** | ❌ | you cannot "un-take" a minimum: remove the current min and the next one is unknowable from the aggregate alone |

The pattern: an aggregate slides when its operation has an **inverse**. Addition has subtraction. Multiplication has division, *except* by zero. `min` has nothing — which is why the sliding-window-maximum problem needs a monotonic deque rather than a running value.

**The product's zero problem, concretely.** Window `[2, 0, 3]` has product 0. Slide to `[0, 3, 4]` and you would compute `0 / 2 * 4 = 0` — correct by luck. But slide `[0, 3, 4]` to `[3, 4, 5]` and you must divide by the departing **0**. The fix is to count the zeros in the window: while that count is above 0 the product is 0, and you maintain the product of the *non-zero* elements separately.

**Simple worked example.** `arr = [1, 2, 3, 4, 5]`, `n = 2`:

| call | window | sum | how |
|---|---|---|---|
| 1 | `[1, 2]` | **3** | computed once in the constructor |
| 2 | `[2, 3]` | **5** | `3 - 1 + 3` |
| 3 | `[3, 4]` | **7** | `5 - 2 + 4` |
| 4 | `[4, 5]` | **9** | `7 - 3 + 5` |

Four windows, and after the first, each cost two arithmetic operations rather than `n` additions.

## Problem Statement

A class taking an array and a window length `n`. Each `next()` returns the aggregate of the current window, then slides right by one.

```python
w = IntegersWindow([1, 2, 3, 4, 5], n=2)
w.next()   # 3   (1+2)
w.next()   # 5   (2+3)
w.next()   # 7   (3+4)
w.next()   # 9   (4+5)
```

**Follow-up 1:** the **product** instead of the sum.
**Follow-up 2:** the **average**.

### Approach 1 — Naive (re-sum the window every call)

**Idea:** on each `next`, slice the window and sum it.

Correct, obvious, and it repeats almost all of its work — of the `n` elements it adds, `n-1` were already counted last time.

**Time complexity:** **O(n) per call**, O(N × n) over a full pass.

**Space complexity:** O(1).

In [ ]:
from collections import deque
from typing import Any, Callable, List, Optional


class NaiveWindow:
    """Baseline: recomputes the whole window on every call."""

    def __init__(self, arr: List[int], n: int) -> None:
        self.arr, self.n, self.start = arr, n, 0

    def next(self) -> int:
        total = sum(self.arr[self.start:self.start + self.n])   # O(n) EVERY call
        if self.start + self.n < len(self.arr):
            self.start += 1
        return total

### Approach 2 — Optimal (a running sum, and an explicit end policy)

**Idea:** repair the sum instead of rebuilding it.

**The design decision the official answer makes by accident.** Its guard — `if start + n < len(arr)` — stops the index from running off the end, and as a side effect makes `next()` **return the last window's sum forever**. A caller looping `while True` gets an endless stream of a stale value with no signal that the data ran out.

That behaviour is defensible; leaving it unstated is not. `on_exhausted` makes it a choice:

- `"repeat"` — the official behaviour, now deliberate.
- `"raise"` — `StopIteration`, which also makes the class a genuine Python iterator, usable directly in a `for` loop.
- `"none"` — a sentinel the caller can test.

**Validate in the constructor.** `n <= 0`, `n > len(arr)`, an empty array: all are errors, and catching them at construction gives a message that names the actual mistake rather than an `IndexError` three calls later.

**Time complexity:** O(n) once to build; **O(1) per call**.

**Space complexity:** O(1).

In [ ]:
class IntegersWindow:
    """Sliding-window sum in O(1) per call, with an explicit end-of-array policy."""

    def __init__(self, arr: List[int], n: int, on_exhausted: str = "repeat") -> None:
        if n <= 0:
            raise ValueError(f"window length must be positive, got {n}")
        if n > len(arr):
            raise ValueError(f"window length {n} exceeds array length {len(arr)}")
        if on_exhausted not in ("repeat", "raise", "none"):
            raise ValueError(f"unknown policy: {on_exhausted!r}")
        self.arr, self.n = arr, n
        self.start = 0
        self.on_exhausted = on_exhausted
        self.current_sum = sum(arr[:n])          # the ONE O(n) pass
        self.exhausted = False

    @property
    def windows_remaining(self) -> int:
        return max(0, len(self.arr) - self.n + 1 - self.start - (1 if self.exhausted else 0))

    def next(self) -> Optional[int]:
        if self.exhausted:                       # the window can no longer slide
            if self.on_exhausted == "raise":
                raise StopIteration
            return None if self.on_exhausted == "none" else self.current_sum

        result = self.current_sum
        if self.start + self.n < len(self.arr):
            # INVARIANT: current_sum == sum(arr[start : start + n]) - repair, do not rebuild
            self.current_sum += self.arr[self.start + self.n] - self.arr[self.start]
            self.start += 1
        else:
            self.exhausted = True                # this was the LAST window
        return result

    # Being a real iterator costs two lines and makes `for s in window` work.
    def __iter__(self):
        return self

    def __next__(self):
        if self.exhausted:
            raise StopIteration
        return self.next()

### Follow-up 1 — the product, and the zero that breaks it

**Idea:** the same slide, with `*` and `/` instead of `+` and `-`. Which works right up until the departing element is **zero**, and `product / 0` raises.

The fix is to split the aggregate in two:

- `nonzero_product` — the product of the window's **non-zero** elements, which always has a valid inverse.
- `zero_count` — how many zeros are in the window.

The answer is then `0 if zero_count else nonzero_product`. Sliding updates whichever of the two the departing and arriving elements belong to. Still **O(1)**, with no recomputation and no special-casing at the call site.

**A floating-point caveat worth stating.** The divide-out trick is exact for integers. For floats it accumulates error, and the error **never washes out**, because each result is computed from the previous one. For float inputs, recompute periodically — or use the deque approach below, which never divides at all.

**Time complexity:** O(1) per call.

**Space complexity:** O(1).

In [ ]:
class ProductWindow:
    """Sliding product, correct in the presence of zeros."""

    def __init__(self, arr: List[int], n: int) -> None:
        if n <= 0 or n > len(arr):
            raise ValueError("invalid window length")
        self.arr, self.n, self.start = arr, n, 0
        self.zero_count = 0
        self.nonzero_product = 1                 # the product of the NON-ZERO elements only
        for v in arr[:n]:
            self._add(v)
        self.exhausted = False

    def _add(self, v: int) -> None:
        if v == 0:
            self.zero_count += 1                 # tracked separately: 0 has no inverse
        else:
            self.nonzero_product *= v

    def _remove(self, v: int) -> None:
        if v == 0:
            self.zero_count -= 1
        else:
            self.nonzero_product //= v if isinstance(v, int) else v

    def next(self) -> int:
        result = 0 if self.zero_count else self.nonzero_product
        if not self.exhausted:
            if self.start + self.n < len(self.arr):
                self._remove(self.arr[self.start])
                self._add(self.arr[self.start + self.n])
                self.start += 1
            else:
                self.exhausted = True
        return result


class AverageWindow(IntegersWindow):
    """The average is the sum, divided. The sliding mechanism does not change at all."""

    def next(self) -> Optional[float]:
        total = super().next()
        return None if total is None else total / self.n

### Follow-up 2 — a stream, with no random access

**Idea:** the running sum needs `arr[start]` — the element **leaving** — which a stream cannot give you, because it has already gone past.

So keep the window itself in a **deque**: push each arriving element, pop the departing one, and it is the pop that hands back the value to subtract. `collections.deque` does both ends in O(1); a list would pay O(n) on `pop(0)`.

This also answers the min/max question. `min` cannot slide, but a deque of *candidates* can: keep indices in decreasing order of value, discard any that a newly arrived larger element makes irrelevant, and the front is always the window maximum. Amortised O(1), because each index is pushed and popped at most once.

**Time complexity:** O(1) amortised per element.

**Space complexity:** **O(n)** — the window, not the stream.

In [ ]:
class StreamingWindow:
    """Sliding sum over a stream: the deque holds the window, so nothing needs re-reading."""

    def __init__(self, n: int) -> None:
        if n <= 0:
            raise ValueError("window length must be positive")
        self.n = n
        self.window: deque = deque()
        self.total = 0

    def push(self, value: int) -> Optional[int]:
        """Feed one element. Returns the window sum once the window is full."""
        self.window.append(value)
        self.total += value
        if len(self.window) > self.n:
            self.total -= self.window.popleft()  # the pop RETURNS the departing value
        return self.total if len(self.window) == self.n else None


def sliding_maximum(arr: List[int], n: int) -> List[int]:
    """max cannot slide as a running value - but a monotonic deque of candidates can."""
    dq: deque = deque()                          # indices, values decreasing front to back
    out: List[int] = []
    for i, v in enumerate(arr):
        while dq and arr[dq[-1]] <= v:
            dq.pop()                             # anything smaller can never be the max again
        dq.append(i)
        if dq[0] <= i - n:
            dq.popleft()                         # the front has slid out of the window
        if i >= n - 1:
            out.append(arr[dq[0]])               # the front is always the window maximum
    return out

## Verification

The worked example, the boundary policies, and the zero case that breaks a naive sliding product.

In [ ]:
import random

# --- The worked example ---
w = IntegersWindow([1, 2, 3, 4, 5], n=2)
assert [w.next() for _ in range(4)] == [3, 5, 7, 9]

# --- Agreement with the naive implementation ---
for arr, n in [([1, 2, 3, 4, 5], 2), ([1, 2, 3, 4, 5], 1), ([1, 2, 3, 4, 5], 5),
               ([5], 1), ([-3, 0, 7, -1], 2), ([0, 0, 0], 2)]:
    a, b = IntegersWindow(arr, n), NaiveWindow(arr, n)
    windows = len(arr) - n + 1
    assert [a.next() for _ in range(windows)] == [b.next() for _ in range(windows)], (arr, n)

# --- The exact window contents, checked against a slice ---
arr = [4, -2, 7, 0, 1, 9, -5]
for n in range(1, len(arr) + 1):
    w = IntegersWindow(arr, n)
    for start in range(len(arr) - n + 1):
        assert w.next() == sum(arr[start:start + n]), (n, start)

# --- THE unstated boundary: three explicit policies ---
r = IntegersWindow([1, 2, 3], n=2, on_exhausted="repeat")
assert [r.next() for _ in range(5)] == [3, 5, 5, 5, 5], "the official behaviour, now deliberate"

nn = IntegersWindow([1, 2, 3], n=2, on_exhausted="none")
assert [nn.next() for _ in range(4)] == [3, 5, None, None]

ex = IntegersWindow([1, 2, 3], n=2, on_exhausted="raise")
assert ex.next() == 3 and ex.next() == 5
try:
    ex.next()
except StopIteration:
    pass
else:
    raise AssertionError("the 'raise' policy must signal exhaustion")

# ...which also makes it a real Python iterator
assert list(IntegersWindow([1, 2, 3, 4, 5], n=2)) == [3, 5, 7, 9], "usable in a for loop"
assert list(IntegersWindow([1, 2, 3], n=3)) == [6], "a single window"

# --- windows_remaining ---
w = IntegersWindow([1, 2, 3, 4, 5], n=2)
assert w.windows_remaining == 4
w.next()
assert w.windows_remaining == 3
[w.next() for _ in range(3)]
assert w.windows_remaining == 0

# --- Constructor validation ---
for arr, n, why in [([1, 2, 3], 0, "zero"), ([1, 2, 3], -1, "negative"),
                    ([1, 2, 3], 4, "larger than the array"), ([], 1, "empty array")]:
    try:
        IntegersWindow(arr, n)
    except ValueError:
        pass
    else:
        raise AssertionError(f"a window length that is {why} must be rejected")

# --- Negative numbers and zeros in the sum ---
w = IntegersWindow([-5, 3, -2, 8], n=2)
assert [w.next() for _ in range(3)] == [-2, 1, 6]
w = IntegersWindow([0, 0, 0, 0], n=3)
assert [w.next() for _ in range(2)] == [0, 0]

# --- Follow-up: the product, including the zero that breaks naive division ---
p = ProductWindow([1, 2, 3, 4], n=2)
assert [p.next() for _ in range(3)] == [2, 6, 12]

# A zero moving THROUGH the window - the case where product/leaving would divide by zero
p = ProductWindow([2, 0, 3, 4], n=2)
assert [p.next() for _ in range(3)] == [0, 0, 12], (
    "the window [3,4] must recover the true product AFTER the zero leaves"
)
p = ProductWindow([0, 0, 5, 6], n=2)
assert [p.next() for _ in range(3)] == [0, 0, 30]
p = ProductWindow([1, 0, 1], n=3)
assert p.next() == 0
p = ProductWindow([-2, 3, -4], n=2)
assert [p.next() for _ in range(2)] == [-6, -12], "signs must survive the divide-out"

# Cross-check the product against a brute-force recomputation
random.seed(131)
for _ in range(400):
    a = [random.randint(-4, 4) for _ in range(random.randint(1, 12))]
    n = random.randint(1, len(a))
    pw = ProductWindow(a, n)
    for start in range(len(a) - n + 1):
        expected = 1
        for v in a[start:start + n]:
            expected *= v
        assert pw.next() == expected, (a, n, start)

# --- Follow-up: the average ---
av = AverageWindow([1, 2, 3, 4, 5], n=2)
assert [av.next() for _ in range(4)] == [1.5, 2.5, 3.5, 4.5]
av = AverageWindow([1, 2, 4], n=3)
assert av.next() == 7 / 3, "a float, not a truncated integer"

# --- Follow-up: a stream, with no random access ---
s = StreamingWindow(n=3)
out = [s.push(v) for v in [1, 2, 3, 4, 5]]
assert out == [None, None, 6, 9, 12], "no sum until the window is full"
s = StreamingWindow(n=1)
assert [s.push(v) for v in [7, 8]] == [7, 8]

# The streaming version must agree with the array version
random.seed(137)
for _ in range(300):
    a = [random.randint(-20, 20) for _ in range(random.randint(1, 25))]
    n = random.randint(1, len(a))
    sw = StreamingWindow(n)
    streamed = [x for x in (sw.push(v) for v in a) if x is not None]
    arrayed = list(IntegersWindow(a, n, on_exhausted="raise"))
    assert streamed == arrayed, (a, n)

# --- max does not slide as a running value, but a monotonic deque handles it ---
assert sliding_maximum([1, 3, -1, -3, 5, 3, 6, 7], 3) == [3, 3, 5, 5, 6, 7]
assert sliding_maximum([9, 8, 7], 2) == [9, 8]
assert sliding_maximum([1, 2, 3], 1) == [1, 2, 3]
for _ in range(300):
    a = [random.randint(-20, 20) for _ in range(random.randint(1, 25))]
    n = random.randint(1, len(a))
    expected = [max(a[i:i + n]) for i in range(len(a) - n + 1)]
    assert sliding_maximum(a, n) == expected, (a, n)

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **A changing `n`.** Growing or shrinking the window means the running sum no longer matches any window, so it has to be rebuilt — O(n). If `n` changes often, the right structure is a **prefix-sum array** instead: `sum(arr[i:j]) = prefix[j] - prefix[i]` answers *any* window in O(1) with no state at all, at the cost of O(N) memory and an O(N) rebuild on every mutation. That is exactly the trade in [`22. Billing_System`](../22.%20Billing_System/22.%20Billing_System.ipynb): a running aggregate suits a *fixed* window walked in order; a prefix sum suits *arbitrary* ranges queried out of order.
- **Sum and product together.** Just keep both aggregates and update both on each slide — they are independent. The general shape is a list of `(add, remove)` pairs, one per aggregate, driven by a single slide loop. Worth noting which aggregates can join that list: only the invertible ones.
- **Huge arrays with limited memory.** The sliding window is already the answer — `StreamingWindow` holds `n` elements, never `N`. This is why the pattern matters far beyond arrays: it is how you compute a moving average over a metrics stream, or a rolling rate limit, without storing history.
- **A circular buffer instead of a deque.** A fixed array of size `n` with a wrapping write index gives O(1) push and implicit eviction, with no allocation per element and much better cache locality. It is what you would use in embedded or high-throughput code; `deque` is the same idea with the bookkeeping done for you.
- **Overflow.** Python integers are arbitrary-precision, so the running sum cannot overflow — but in Java or C++ it can, and the sliding *product* overflows spectacularly fast. Use a wider type, or track the product's logarithm if you only need to compare magnitudes.
- **The deeper pattern: which aggregates slide.** An aggregate supports incremental removal exactly when its operation has an **inverse**. Sum has subtraction; product has division (except by zero, hence the zero counter); **min and max have nothing**, which is why `sliding_maximum` keeps a monotonic deque of *candidates* rather than a running value. Being able to say *why* min is different — rather than just knowing the deque trick — is the thing worth taking from this problem.

## Empirical complexity check

Compare **re-summing each window** with the **running sum**, over a full pass across a growing array with a window of fixed proportion.

| Growth when the array doubles | What it means |
|---|---|
| ~4x | quadratic — O(N) windows, each costing O(n), and n grows with N |
| ~2x | linear — O(N) windows at O(1) each |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random


def make_array(n):
    rng = random.Random(139)
    arr = [rng.randint(-100, 100) for _ in range(n)]
    return (arr, max(1, n // 20))            # the window grows WITH the array


def run_naive(arr, window):
    w = NaiveWindow(arr, window)
    for _ in range(len(arr) - window + 1):
        w.next()                             # O(window) each => O(N * window)


def run_sliding(arr, window):
    w = IntegersWindow(arr, window)
    for _ in range(len(arr) - window + 1):
        w.next()                             # O(1) each => O(N)


benchmark(
    {"Approach 1 - re-sum each window": run_naive,
     "Approach 2 - running sum O(1)/call": run_sliding},
    make_array,
    sizes=[500, 1000, 2000, 4000],
    repeats=2,
)

## Patterns learned

- **Consecutive windows overlap, so repair the aggregate instead of rebuilding it.** One element leaves, one arrives; everything between is unchanged. That single observation is the sliding window pattern.
- **State the invariant before writing the loop.** *"`current_sum` is always the sum of `arr[start : start+n]`."* The correctness argument is then two sentences, and the off-by-ones become obvious.
- **An aggregate can slide only if its operation has an inverse.** Sum has subtraction; product has division, except by zero; min and max have nothing at all. That test tells you immediately whether a running value will work or whether you need a monotonic deque.
- **Handle the special value structurally, not with a special case.** Counting zeros separately from the non-zero product keeps the slide branch-free and the call site clean — far better than checking for zero at every use.
- **Decide the boundary; do not let a guard decide it for you.** "Repeat the last value forever" is a defensible policy and a terrible accident. Make it a parameter, and get `StopIteration` — and a real Python iterator — almost for free.
- **A stream cannot look backwards, so keep the window itself.** The `deque` pop hands back the departing value that random access would have supplied. O(n) memory in the *window*, not the stream.
- **Running aggregate versus prefix sum.** A running value suits a fixed window walked in order; a prefix array suits arbitrary ranges queried out of order. Knowing which question you are answering picks the structure.